# Local LLM Lab

Run a real open-source LLM on your own laptop, look inside it to understand how it works, then
experiment with fine-tuning. The model is **Qwen2.5-3B-Instruct** — a capable ~3B-parameter open model
that fits on 16 GB of RAM.

**Honest expectations.** This is a learning lab, not a replacement for hosted frontier models. A 3B model
on a CPU will answer basic questions, summarise, and chat — but slowly, and it will sometimes be
confidently wrong. Seeing *where* it breaks is part of the learning.

### The three paths
1. **Understand** (this notebook, `transformers`) — slow, but lets you see the tokenizer, the architecture,
   the raw next-token probabilities, and the generation loop.
2. **Chat fast** (Ollama) — a quantized build that's responsive enough for everyday use (cell near the end).
3. **Fine-tune** (Colab) — train a small LoRA adapter on a free GPU, then run it here.

Run the cells top to bottom. The first model-load cell downloads ~6 GB (cached afterwards).

## 1. Transformer vs RNN (clearing up a common mix-up)

Transformers are **not** a kind of RNN — they *replaced* RNNs for language.

- An **RNN** reads tokens one at a time, carrying a hidden "memory" vector forward. Sequential, hard to
  parallelise, and it tends to forget things from far back.
- A **Transformer** sees all tokens at once and uses **attention**: every token directly looks at every
  other token and decides how much each one matters. That parallelism + direct long-range access is why
  Transformers scaled into modern LLMs and RNNs didn't.

Everything below is a Transformer (Qwen2.5).

## 2. Environment check

Confirm we're on CPU, see how much RAM we have, and set the thread count. (No NVIDIA GPU here, so
`device` will be `cpu` — that's expected.)

In [ ]:
import os, platform
import torch

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # the model we use everywhere below

print("Python    :", platform.python_version())
print("PyTorch   :", torch.__version__)
print("CUDA avail:", torch.cuda.is_available(), "(False is expected on this laptop)")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device    :", device)
print("CPU cores :", os.cpu_count())

try:
    import psutil
    gb = psutil.virtual_memory().total / 1e9
    print(f"Total RAM : {gb:.1f} GB")
except Exception:
    pass

# CPU inference benefits from a sensible thread count. Tune if you like.
torch.set_num_threads(max(1, (os.cpu_count() or 4) // 2))
print("Torch threads:", torch.get_num_threads())

## 3. The model: Qwen2.5-3B-Instruct

Qwen2.5 is a family of open models from Alibaba Cloud. The **3B Instruct** variant is small enough to run
here and is *instruction-tuned* (it's been trained to follow chat-style prompts). It's **ungated** — no
license signup needed to download it.

## 4. The tokenizer — how text becomes numbers

A model never sees text; it sees integer **token IDs**. The tokenizer splits text into subword pieces and
maps each to an ID. Let's load it and look.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)

text = "Hello! How do transformers actually work?"
ids = tok(text)["input_ids"]

print("Text       :", text)
print("Token IDs  :", ids)
print("Tokens     :", tok.convert_ids_to_tokens(ids))
print("Round-trip :", tok.decode(ids))
print("Vocab size :", tok.vocab_size)

# Notice how common words are one token but rarer/sub-word chunks split apart.
# The leading characters (e.g. 'Ġ') mark a preceding space in this tokenizer.

## 5. Load the model and look at its architecture

We load in **bfloat16** (~6 GB) so it fits comfortably in 16 GB RAM. The first run downloads the weights
from Hugging Face (~6 GB) — be patient; it's cached after that.

Then we print the config and the module tree so you can *see* the Transformer: a stack of decoder layers,
each with self-attention + an MLP.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
model.eval()

cfg = model.config
print("hidden size      :", cfg.hidden_size)
print("num layers       :", cfg.num_hidden_layers)
print("attention heads  :", cfg.num_attention_heads)
print("KV heads (GQA)    :", getattr(cfg, "num_key_value_heads", "n/a"))
print("vocab size       :", cfg.vocab_size)

n_params = sum(p.numel() for p in model.parameters())
print(f"\ntotal parameters : {n_params/1e9:.2f} B")

# The module tree: embedding -> N decoder layers (attention + MLP) -> norm -> lm_head
print()
print(model)

## 6. How generation works: one token at a time

An LLM is just a **next-token predictor**. Given a sequence, it outputs a score (logit) for every token in
the vocabulary; softmax turns those into probabilities. To generate text, you pick a token, append it, and
repeat.

Let's do a single forward pass and look at the model's top candidates for the *next* token.

In [ ]:
prompt = "The capital of France is"
inputs = tok(prompt, return_tensors="pt")

with torch.no_grad():
    out = model(**inputs)

# logits for the position right after the prompt
next_logits = out.logits[0, -1]
probs = torch.softmax(next_logits.float(), dim=-1)
topk = torch.topk(probs, 10)

print(f"Prompt: {prompt!r}\nTop-10 next-token candidates:\n")
for score, idx in zip(topk.values, topk.indices):
    print(f"  {tok.decode([idx])!r:>15}  p={score.item():.4f}")

## 7. Full generation

Now we let the model generate a full answer with `model.generate`. We use the **chat template** so the
prompt is formatted exactly the way the instruct model expects.

> On CPU this runs at only a few tokens/sec, so we keep `max_new_tokens` small while exploring. This cell
> may take a little while — that's normal.

In [ ]:
messages = [{"role": "user", "content": "In one sentence, what is a transformer in machine learning?"}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")

with torch.no_grad():
    gen = model.generate(inputs, max_new_tokens=60, do_sample=False)

answer = tok.decode(gen[0, inputs.shape[1]:], skip_special_tokens=True)
print(answer)

## 8. A small multi-turn chat helper

Wrapping the above in a function gives us a basic conversation loop that keeps history. (Each turn
re-processes the whole history, so keep conversations short on CPU.)

In [ ]:
def chat(history, user_msg, max_new_tokens=128):
    """Append a user turn, generate a reply, return (new_history, reply)."""
    history = history + [{"role": "user", "content": user_msg}]
    inputs = tok.apply_chat_template(history, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        gen = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = tok.decode(gen[0, inputs.shape[1]:], skip_special_tokens=True)
    return history + [{"role": "assistant", "content": reply}], reply

history = []
history, reply = chat(history, "Hi! In one short sentence, what can you help me with?")
print("Assistant:", reply)

history, reply = chat(history, "What did I just ask you?")
print("\nAssistant:", reply)

## 9. The fast path: Ollama

`transformers` on CPU is great for *looking inside* the model, but slow for everyday chatting. For that,
use **[Ollama](https://ollama.com)**, which runs a 4-bit quantized build (~2 GB) much faster.

1. Install Ollama from <https://ollama.com/download>.
2. In a terminal: `ollama run qwen2.5:3b` (first run downloads the model).
3. Ollama serves an HTTP API on `localhost:11434`. The cell below uses it, and skips gracefully if Ollama
   isn't installed/running.

In [ ]:
import requests

def ollama_chat(prompt, model="qwen2.5:3b"):
    try:
        r = requests.post(
            "http://localhost:11434/api/chat",
            json={"model": model,
                  "messages": [{"role": "user", "content": prompt}],
                  "stream": False},
            timeout=180,
        )
        r.raise_for_status()
        return r.json()["message"]["content"]
    except Exception as e:
        return ("(Ollama not reachable: %s)\n"
                "Install from https://ollama.com and run:  ollama run qwen2.5:3b" % e)

print(ollama_chat("Give me one fun fact about octopuses."))

## 10. Fine-tuning — the idea

**Fine-tuning** nudges the model's behaviour using your own examples. We use **LoRA** (Low-Rank
Adaptation): the big model stays frozen and we train a tiny set of extra weights "on top". The resulting
**adapter** is only a few MB.

Two key points:
- **It teaches style/tone/format far more reliably than new facts.** Our example dataset teaches the model
  to always finish with a `TL;DR:` line.
- **Training needs a GPU**, which this laptop doesn't have — so we train on a **free Google Colab GPU**
  (QLoRA), then run the adapter here on CPU. See `finetune/README.md`.

## 11. Build your fine-tuning dataset

Each training example is a `{"messages": [...]}` record with a user turn and the assistant reply you *wish*
the model would give. Use `add_example(...)` to append to `finetune/data/train.jsonl`, then preview it.

In [ ]:
import json

DATA_PATH = "../finetune/data/train.jsonl"   # relative to this notebook (notebooks/)

def add_example(user, assistant, path=DATA_PATH):
    rec = {"messages": [{"role": "user", "content": user},
                        {"role": "assistant", "content": assistant}]}
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    n = sum(1 for _ in open(path, encoding="utf-8"))
    print(f"Added. Dataset now has {n} examples.")

# --- Add your own (uncomment, edit, re-run) ---
# add_example("What's 2 + 2?", "2 + 2 = 4.\n\nTL;DR: 4.")

# --- Preview the current dataset ---
with open(DATA_PATH, encoding="utf-8") as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        u = rec["messages"][0]["content"]
        print(f"{i:>2}  USER: {u[:70]}")

## 12. Train on Colab, then come back

1. Open `finetune/colab_finetune.ipynb` in **Google Colab** (set runtime to **T4 GPU**).
2. Upload your `train.jsonl` when prompted, run all cells.
3. Download the produced adapter zip and unzip it into
   `models/adapters/qwen2.5-3b-tldr/` in this project.

Full instructions: `finetune/README.md`. Then run the next cell.

## 13. Run your fine-tuned model locally

Once you've trained and unzipped an adapter, this loads the base model + your adapter with `peft` and
generates with it. Until then, it prints a friendly reminder and skips.

In [ ]:
ADAPTER_DIR = "../models/adapters/qwen2.5-3b-tldr"

if not os.path.isdir(ADAPTER_DIR):
    print("No adapter yet at", ADAPTER_DIR)
    print("Train one on Colab (see finetune/README.md), unzip it there, then re-run this cell.")
else:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
    ft = PeftModel.from_pretrained(base, ADAPTER_DIR)
    ft.eval()

    msgs = [{"role": "user", "content": "What is the capital of France?"}]
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        g = ft.generate(inp, max_new_tokens=80, do_sample=False)
    print("FINE-TUNED reply:\n")
    print(tok.decode(g[0, inp.shape[1]:], skip_special_tokens=True))
    print("\n(Look for the trained 'TL;DR:' sign-off.)")

## 14. Where to go next

- **Sampling**: try `do_sample=True, temperature=0.7, top_p=0.9` in `generate` and watch variety change.
- **Bigger context**: feed a paragraph and ask for a summary.
- **More data**: add 30-50 consistent examples and re-train — the style effect gets much stronger.
- **Quantization**: compare `transformers` (bf16) vs Ollama (4-bit) speed and quality.
- **Read the source**: `transformers` model code for Qwen lives in your `.venv` under
  `transformers/models/qwen2/` — open `modeling_qwen2.py` to see attention + MLP implemented.